# Pruebas con geolocalización y mapas

In [38]:
import pandas as pd
import os
import sys

import folium
from IPython.display import IFrame

# Ruta a la raíz del proyecto
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

from utils.ui_data import DEPARTAMENTOS_CIUDADES

In [39]:
from geopy.geocoders import Nominatim


def get_coordinates(city_name: str):
    """Return latitude and longitude from a city name using OpenStreetMap's Nominatim."""
    geolocator = Nominatim(user_agent="triage_app")
    location = geolocator.geocode("ciudad: " + city_name + ", Colombia")
    if location:
        return (location.latitude, location.longitude)
    else:
        location = geolocator.geocode(city_name + ", Colombia")
        if location:
            return (location.latitude, location.longitude)

## Centrar mapa en base a nombde de ciudad/municipio

In [51]:
# Example:
coords = get_coordinates("Villavicencio, Meta")
print(coords)  # → (6.244203, -75.581212)

(4.1223121, -73.6286146)


In [52]:
m = folium.Map(location=coords, zoom_start=14)

m.save("mapa.html")  # Guardar el mapa en un archivo HTML


In [ ]:
import requests
import streamlit as st


# Usamos st.cache_data para no volver a consultar la misma dirección dos veces
# Esto evita que te bloqueen por exceso de peticiones y hace la app más rápida.
@st.cache_data(show_spinner=False)
def geocode_address_arcgis(address: str):
    """
    Geocoding: Dirección -> Coordenadas
    """
    url = "https://geocode.arcgis.com/arcgis/rest/services/World/GeocodeServer/findAddressCandidates"

    # Es buena práctica limpiar la dirección
    clean_address = address.strip()
    if not clean_address:
        return None

    params = {
        "f": "json",
        "singleLine": f"{clean_address}, Colombia",  # Forzamos búsqueda en Colombia
        "maxLocations": 1,
        "outFields": "Match_addr,Addr_type",
    }

    try:
        response = requests.get(url, params=params, timeout=5)
        response.raise_for_status()  # Lanza error si hay problemas de conexión
        data = response.json()

        if data.get("candidates"):
            top = data["candidates"][0]
            lat = top["location"]["y"]
            lon = top["location"]["x"]
            formatted = top["address"]
            return {"lat": lat, "lon": lon, "address": formatted}

    except Exception as e:
        st.error(f"Error conectando con el servicio de mapas: {e}")
        return None

    return None


@st.cache_data(show_spinner=False)
def reverse_geocode_arcgis(lat: float, lon: float):
    """
    Reverse Geocoding: Coordenadas -> Dirección aproximada
    """
    url = "https://geocode.arcgis.com/arcgis/rest/services/World/GeocodeServer/reverseGeocode"

    params = {
        "f": "json",
        "location": f"{lon},{lat}",  # Nota: ArcGIS usa x,y (lon,lat)
        "distance": 100,  # Buscar en un radio de 100 metros
        "outSR": "",
    }

    try:
        response = requests.get(url, params=params, timeout=5)
        if response.status_code == 200:
            data = response.json()
            if "address" in data:
                return data["address"]["Match_addr"]
    except Exception:
        return None

    return "Dirección no encontrada"

2025-12-02 15:48:12.685 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2025-12-02 15:48:12.685 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [ ]:
geocode_address_arcgis("Calle 30 11 100-100, Yopal, Casanare")

{'lat': 5.327470377273,
 'lon': -72.401158999544,
 'address': 'Calle 30 11 100, Yopal, Casanare'}

In [62]:
reverse_geocode_arcgis(5.326766, -72.401931)


'Calle 30 11 100-100, Yopal, Casanare'